# 06 — Advanced: distances & choosing k

*Module 01 · k-Nearest Neighbours — notebook 6 of 6 (optional, advanced)*

You have a complete, working command of k-NN. This optional notebook is a deeper, longer dive for the
curious, built around pictures — because its subject, the **geometry of distance**, is best *seen*.

We answer two questions you may be asking. **First: does the *definition* of distance matter?** We
draw the shapes that different metrics carve out, watch them reshape k-NN's decision boundary, meet
**Mahalanobis** (a distance that learns the data's shape) and **cosine**, and find *when* the choice
actually changes predictions — testing it on both of the course's datasets. **Second: how do you
choose k rigorously?** We meet **nested cross-validation** and see why a tuned score flatters itself.

**Prerequisites:** notebooks 01–05 (the whole k-NN chapter).

**What you will be able to do**

- See a metric as the *shape* of a neighbourhood, and watch that shape reach the decision boundary.
- Recognize Minkowski p, Mahalanobis, and cosine distance, and what each is for.
- Explain *when* the choice of metric matters — barely in low dimensions, but in high ones.
- Choose k honestly with nested cross-validation, and say why a tuned CV score is optimistic.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    cross_val_score,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course import datasets, viz
from ml_course.colors import CLASS_CYCLE, COLORS

# Fix the seed so every run tells the same story.
np.random.seed(0)
viz.use_course_style()

# Two 2-D sets for geometry; one high-dimensional set for the curse. Standardize the 2-D sets
# (notebook 2), so what we see below is about the metric, not the scale.
X_moons, y_moons = make_moons(n_samples=300, noise=0.30, random_state=0)
X_moons = StandardScaler().fit_transform(X_moons)

X_peng, y_peng = datasets.penguins_xy()
X_peng = StandardScaler().fit_transform(X_peng)
y_peng = (y_peng.to_numpy() == "Gentoo").astype(int)

bc = load_breast_cancer()

print(f"moons: {X_moons.shape}   penguins: {X_peng.shape}   breast cancer: {bc.data.shape}")

## Part 1 — A distance is a choice, with a geometry

Throughout the chapter, "distance" meant Euclidean (straight-line, L2). It is the default, not the
only choice. The **Minkowski family** is parameterized by a number `p`:

$$d_p(a, b) = \left( \sum_i |a_i - b_i|^p \right)^{1/p}$$

`p = 2` is Euclidean; `p = 1` is **Manhattan** (city-block, from notebook 2); `p \to \infty` is
**Chebyshev** (the largest single-coordinate gap). The cleanest way to *see* a metric is to draw its
**unit ball** — every point exactly distance 1 from the centre.

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
directions = np.c_[np.cos(theta), np.sin(theta)]

fig, ax = plt.subplots(figsize=(6, 6))
for (p, label), color in zip(
    [(1, "p = 1 (Manhattan)"), (2, "p = 2 (Euclidean)"), (np.inf, "p = inf (Chebyshev)")],
    CLASS_CYCLE,
    strict=False,
):
    if np.isinf(p):
        norm = np.abs(directions).max(axis=1)
    else:
        norm = (np.abs(directions) ** p).sum(axis=1) ** (1 / p)
    ball = directions / norm[:, None]
    ax.plot(ball[:, 0], ball[:, 1], color=color, linewidth=2.5, label=label)

ax.set_aspect("equal")
ax.axhline(0, color=COLORS["muted"], linewidth=0.6)
ax.axvline(0, color=COLORS["muted"], linewidth=0.6)
ax.set_title("Unit balls: every point at distance 1 from the centre")
ax.legend(loc="upper right")
plt.show()

**Read the figure.** Same "distance 1", three shapes: Manhattan's unit ball is a **diamond**,
Euclidean's the familiar **circle**, Chebyshev's a **square**. The unit ball is the set of points a
metric counts as "within distance 1" — so a metric is, literally, the shape of a neighbourhood.

## From the ball to the boundary

That shape does not stay abstract: it propagates into k-NN's **decision boundary**, because the
boundary is built from "who is within reach". To see it we use `make_moons` — its overlapping classes
give the boundary room to show the metric's signature (on cleanly separated data the three would
nearly coincide, a point we return to in Part 3).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (label, kw) in zip(
    axes,
    [
        ("Manhattan (p=1)", {"p": 1}),
        ("Euclidean (p=2)", {"p": 2}),
        ("Chebyshev (p=inf)", {"metric": "chebyshev"}),
    ],
    strict=True,
):
    model = KNeighborsClassifier(n_neighbors=15, **kw).fit(X_moons, y_moons)
    viz.plot_decision_boundary(model, X_moons, y_moons, resolution=200, ax=ax)
    ax.set_title(f"moons - {label}")
fig.tight_layout()
plt.show()

**Read the figure.** Same data, same k = 15 — three different boundaries. Manhattan's leans toward
axis-aligned segments (a staircase *tendency*, sharpest at small k); Euclidean's is smooth; Chebyshev's
is blockier and more diagonal. Each boundary carries the geometry of its unit ball — the diamond, the
circle, the square from the previous figure — so the metric's shape shows up in the decision itself.

## Part 2 — Mahalanobis: a distance that learns the data's shape

All three Minkowski metrics treat every direction alike. But features are often **correlated** — and
standardization (notebook 2) equalized each feature's *scale*, not the correlations between them. Our
penguins are a clear case: longer bills go with longer flippers, so the two features correlate about
**0.87** even after standardizing. **Mahalanobis distance** handles that: it whitens by the covariance
$\Sigma$,

$$d_M(a, b) = \sqrt{(a - b)^\top \Sigma^{-1} (a - b)},$$

so its unit ball is an **ellipse aligned to the data**. Here it is on the standardized penguins.

In [ ]:
Sigma = np.cov(X_peng.T)  # standardized penguins: scale handled (NB 2), correlation ~0.87 remains
t = np.linspace(0, 2 * np.pi, 200)
circle = np.c_[np.cos(t), np.sin(t)]
radius = 2.0
ellipse = (np.linalg.cholesky(Sigma) @ (radius * circle).T).T

fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(X_peng[:, 0], X_peng[:, 1], color=COLORS["muted"], s=14, alpha=0.5, label="penguins")
ax.plot(radius * circle[:, 0], radius * circle[:, 1],
        color=COLORS["train"], linewidth=2.5, label="Euclidean ball")
ax.plot(ellipse[:, 0], ellipse[:, 1],
        color=COLORS["highlight"], linewidth=2.5, label="Mahalanobis ball")
ax.set_aspect("equal")
ax.set_xlabel("bill length (standardized)")
ax.set_ylabel("flipper length (standardized)")
ax.set_title("Euclidean circle vs Mahalanobis ellipse on the penguins")
ax.legend(loc="upper left")
plt.show()

**Read the figure.** Even after standardizing — both axes share the same spread — the penguin cloud
leans along the diagonal: bill and flipper rise together. The Euclidean ball ignores that and stays a
round circle; the Mahalanobis ball is an **ellipse tilted along the cloud**, treating a step *across*
the correlation as farther than a step *along* it. That tilt is the correlation standardization left
behind — and what Mahalanobis adds.

In [ ]:
VI = np.linalg.inv(np.cov(X_peng.T))  # inverse covariance: the whitening Mahalanobis uses
maha = KNeighborsClassifier(
    n_neighbors=15, metric="mahalanobis", metric_params={"VI": VI}, algorithm="brute"
).fit(X_peng, y_peng)

euclid = KNeighborsClassifier(n_neighbors=15).fit(X_peng, y_peng)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))
viz.plot_decision_boundary(euclid, X_peng, y_peng, resolution=200, ax=axes[0])
axes[0].set_title("penguins - Euclidean k-NN")
viz.plot_decision_boundary(maha, X_peng, y_peng, resolution=200, ax=axes[1])
axes[1].set_title("penguins - Mahalanobis k-NN")
for ax in axes:
    ax.set_xlabel("bill length (standardized)")
    ax.set_ylabel("flipper length (standardized)")
fig.tight_layout()
plt.show()

**Read the figure.** Swap Euclidean for Mahalanobis and the k-NN boundary **reshapes** — it bends to
follow the correlated cloud instead of cutting a straight diagonal. Both separate these well-behaved
penguins about equally well; what changed is the *shape* of the decision, which is the metric's doing.
(A fair *accuracy* contest would need a metric fitted to each class's own spread — its own method,
beyond this chapter; it is the idea behind linear discriminant analysis — so here we show the
geometry, not a rigged race.) One more distance worth naming: **cosine distance** compares the *angle*
between vectors, ignoring magnitude — the natural choice for text or word-count data.

## Part 3 — When does the metric matter? It depends on the dimension

On the moons the metric nudged the boundary but rarely the verdict, and on separated data it would
matter even less. So *when* does it bite? In **high dimensions** — where, as notebook 5 showed,
distances **concentrate**. Let us see that concentration directly, then watch the metric's effect grow
with dimension, on both of the course's datasets.

In [ ]:
rng = np.random.default_rng(0)

fig, ax = plt.subplots(figsize=(8, 5))
for d, color in zip((2, 50, 1000), CLASS_CYCLE, strict=False):
    points = rng.standard_normal((300, d))
    dmat = pairwise_distances(points)
    pairs = dmat[np.triu_indices_from(dmat, k=1)]
    ax.hist(pairs / pairs.mean(), bins=60, density=True, histtype="step",
            linewidth=2, color=color, label=f"d = {d}")
ax.set_xlabel("pairwise distance / mean distance")
ax.set_ylabel("density")
ax.set_title("Distances concentrate as the dimension grows")
ax.legend()
plt.show()

**Read the figure.** Each curve is the spread of pairwise distances among random points, divided by
their mean. In 2-D (d = 2) distances range widely — some pairs are 3-4× closer than others, so
"nearest" is meaningful. By d = 1000 the whole distribution has collapsed onto 1: every point is
almost exactly the mean distance from every other. When the nearest neighbour is barely closer than
the farthest, k-NN has nothing to grip. That is the curse of dimensionality, seen directly.

In [ ]:
dims = [2, 20, 100, 500, 1000]
ratios = {p: [] for p in (0.5, 1, 2)}
for d in dims:
    points = rng.standard_normal((200, d))
    for p in (0.5, 1, 2):
        dmat = pairwise_distances(points, metric="minkowski", p=p)
        np.fill_diagonal(dmat, np.nan)
        ratios[p].append(float(np.nanmean(np.nanmin(dmat, 1) / np.nanmax(dmat, 1))))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for p, color in zip((0.5, 1, 2), CLASS_CYCLE, strict=False):
    ax.plot(dims, ratios[p], color=color, marker="o", linewidth=2, label=f"p = {p}")
ax.set_xlabel("dimensions (pure noise)")
ax.set_ylabel("mean near/far distance ratio")
ax.set_title("A smaller p resists concentration (the ratio rises toward 1 later)")
ax.legend()
plt.show()

**Read the figure.** The near/far ratio — the nearest distance over the farthest — climbs toward 1 as
dimensions grow (the concentration from the previous figure). But it climbs *more slowly for a smaller
p*: at every dimension, p = 0.5 < p = 1 < p = 2. So a smaller p keeps a little more contrast in high
dimensions — exactly Aggarwal and colleagues' finding (2001; this ratio is a simpler cousin of their
*relative contrast*, and they argued fractional norms p < 1 hold up best).

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print("penguins (2 features) - k-NN(5) CV accuracy by metric:")
for label, kw in [
    ("Manhattan (p=1)", {"p": 1}),
    ("Euclidean (p=2)", {"p": 2}),
    ("Chebyshev", {"metric": "chebyshev"}),
]:
    acc = cross_val_score(KNeighborsClassifier(5, **kw), X_peng, y_peng, cv=cv).mean()
    print(f"  {label:18s}: {acc:.3f}")

# Breast cancer + noise: Manhattan vs Euclidean, averaged over 8 noise draws.
Xtr, Xte, ytr, yte = train_test_split(
    bc.data, bc.target, test_size=0.30, stratify=bc.target, random_state=0
)
sc = StandardScaler().fit(Xtr)
Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
levels = [0, 50, 100, 200, 500, 1000]
mean_p1, mean_p2 = [], []
for d in levels:
    acc1, acc2 = [], []
    for seed in range(8):
        r = np.random.default_rng(seed)
        if d:
            a_tr = np.hstack([Xtr_s, r.standard_normal((Xtr_s.shape[0], d))])
            a_te = np.hstack([Xte_s, r.standard_normal((Xte_s.shape[0], d))])
        else:
            a_tr, a_te = Xtr_s, Xte_s
        acc1.append(KNeighborsClassifier(7, p=1).fit(a_tr, ytr).score(a_te, yte))
        acc2.append(KNeighborsClassifier(7, p=2).fit(a_tr, ytr).score(a_te, yte))
    mean_p1.append(np.mean(acc1))
    mean_p2.append(np.mean(acc2))

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(levels, mean_p1, color=COLORS["model"], marker="o", linewidth=2, label="Manhattan (p=1)")
ax.plot(levels, mean_p2, color=COLORS["error"], marker="s", linewidth=2, label="Euclidean (p=2)")
ax.set_xlabel("breast cancer: pure-noise features added")
ax.set_ylabel("test accuracy (mean of 8 draws)")
ax.set_title("In high dimensions the smaller p (Manhattan) pulls ahead")
ax.legend()
plt.show()

**Read the figure and output.** Two datasets, one lesson. On the **penguins** (two well-separated
features) the metric is a *wash* — Manhattan, Euclidean, and Chebyshev all score 0.993. On **breast
cancer** the two metrics tie with no noise, then separate as noise dimensions pile up: averaged over
eight draws, Manhattan (p = 1) pulls ahead of Euclidean (p = 2), the gap widening with dimension (to
about 0.84 vs 0.80 at +1000). It is a *modest* effect, and we had to average to see it — on a single
draw the accuracy is noisy. The honest summary: **the metric barely matters in low dimensions but
matters, modestly, in high ones.** And keep it in proportion — notebook 5's bigger lever still stands:
the surest fix for the curse is fewer dimensions, not a different p.

## Part 4 — Choosing k rigorously: nested cross-validation

We chose k by cross-validation (notebook 3). One subtlety remains, about *reporting* that choice.
Cross-validate k = 1, 3, 5, … and quote the best score, and that number is **optimistic**: by keeping
the *best* of many k's, you let the validation folds shape the choice, and the winner partly reflects
luck (a "winner's curse"). The fix is to **nest** the procedure — an inner cross-validation chooses k,
and an outer one, on data the tuning never saw, estimates performance.

In [ ]:
fig, (ax_outer, ax_inner) = plt.subplots(2, 1, figsize=(9, 4.2))
for i in range(5):
    held = i == 2
    ax_outer.add_patch(Rectangle((i, 0), 0.94, 1, edgecolor=COLORS["background"],
                                 facecolor=COLORS["test"] if held else COLORS["train"]))
    ax_outer.text(i + 0.47, 0.5, "estimate" if held else "train",
                  ha="center", va="center", color=COLORS["background"], fontsize=9)
ax_outer.set(xlim=(0, 5), ylim=(0, 1))
ax_outer.set_title("Outer CV: each fold is held out in turn (honest estimate)")
ax_outer.axis("off")

for i in range(5):
    val = i == 1
    ax_inner.add_patch(Rectangle((i, 0), 0.94, 1, edgecolor=COLORS["background"],
                                 facecolor=COLORS["highlight"] if val else COLORS["model"]))
    ax_inner.text(i + 0.47, 0.5, "validate" if val else "fit",
                  ha="center", va="center", color=COLORS["background"], fontsize=9)
ax_inner.set(xlim=(0, 5), ylim=(0, 1))
ax_inner.set_title("Inner CV (within each outer-train block): choose k")
ax_inner.axis("off")
fig.tight_layout()
plt.show()

**Read the figure.** Two nested loops. The **outer** loop holds out each fold in turn and never lets
it touch the tuning — those held-out folds give the honest estimate. **Inside** each outer-train
block, an **inner** loop does its own cross-validation to pick k. Because the k that wins is chosen
without ever seeing the outer fold it is judged on, the estimate is not inflated by the choice.

In [ ]:
pipe = make_pipeline(StandardScaler(), KNeighborsClassifier())
grid = {"kneighborsclassifier__n_neighbors": [1, 3, 5, 7, 9, 11, 15, 21, 31]}
inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

naive = GridSearchCV(pipe, grid, cv=inner).fit(Xtr, ytr).best_score_
outer_scores = cross_val_score(GridSearchCV(pipe, grid, cv=inner), Xtr, ytr, cv=outer)
nested = outer_scores.mean()
print(f"naive (best inner-CV score, optimistic): {naive:.3f}")
print(f"nested CV estimate: {nested:.3f} +/- {outer_scores.std():.3f}")

fig, ax = plt.subplots(figsize=(7.5, 3.6))
y_jitter = (np.arange(len(outer_scores)) - (len(outer_scores) - 1) / 2) * 0.18
ax.scatter(outer_scores, y_jitter, color=COLORS["model"], s=130, zorder=3,
           label="outer-fold scores")
ax.axvline(nested, color=COLORS["correct"], linewidth=2, label=f"nested mean = {nested:.3f}")
ax.axvline(naive, color=COLORS["error"], linewidth=2, linestyle="--",
           label=f"naive (optimistic) = {naive:.3f}")
ax.set_ylim(-0.7, 0.7)
ax.set_yticks([])
ax.set_xlabel("accuracy")
ax.set_title("The tuned score sits above the honest nested estimate")
ax.legend(loc="upper left")
plt.show()

**Read the figure.** The five dots are the outer-fold accuracies — and they spread from about 0.94 to
0.99, real fold-to-fold variation a single number would hide. The honest nested estimate is their mean
(0.960); the naive "best of the grid" (0.967) sits above it, optimistic. The gap is small here and
grows with bigger grids and smaller data, but the shape of the lesson is fixed: tune in an inner loop,
estimate in an outer one. The simplest honest estimate of all remains notebook 5's — tune on the
training set, then look at a sealed test set exactly once.

## Your turn

1. You cross-validate ten values of k and report the best score as your model's accuracy. Why is that
   number optimistic, and what does nested cross-validation do about it?
2. The metric was a wash on the penguins but mattered on the noisy breast-cancer data. In one
   sentence, what decides whether the choice of metric matters?
3. You face a problem with 500 features. Which Minkowski p would you try first, and why — and what
   would help even more than changing p (notebook 5)?
4. The Mahalanobis boundary looked clearly different from the Euclidean one on the penguins, yet their
   accuracies are about the same. How can the decision boundary change while the accuracy barely does?

## What you built

You went past the defaults, and you *saw* why they are defaults. A metric is the shape of a
neighbourhood — diamond, circle, or square — and that shape reaches all the way into k-NN's decision
boundary. Mahalanobis reshapes the neighbourhood to the data's covariance (its ellipse tilted along
the correlated penguins, and its boundary with it); cosine compares direction alone. You watched
distances **concentrate** in high dimensions until "nearest" lost meaning, saw a smaller p resist that
a little, and learned the honest summary: the metric is a wash in low dimensions but matters, modestly,
in high ones. Finally you sharpened how you *report* a tuned k — nested cross-validation, which does
not flatter itself.

That closes the k-Nearest Neighbours chapter. You built the method by hand, fixed its scale, tuned its
dial, met its estimator, ran it honestly on real data, felt its limits — and now you have seen what
lies one step beyond.

**Vocabulary**

- **Minkowski p / unit ball** — the family of distances (p=1 Manhattan, 2 Euclidean, ∞ Chebyshev); the
  unit ball is the shape of a neighbourhood, and it reaches the decision boundary.
- **Mahalanobis distance** — covariance-aware distance; its unit ball is an ellipse aligned to the data.
- **cosine distance** — compares direction, ignoring magnitude.
- **distance concentration / metric × dimension** — in high dimensions distances bunch together; the
  metric barely matters in low dimensions but matters (a little) in high ones, where smaller p helps.
- **nested cross-validation** — inner CV tunes, outer CV estimates; avoids the optimism of reporting a
  tuned score.

## References

- C. C. Aggarwal, A. Hinneburg, D. A. Keim (2001). *On the surprising behavior of distance metrics in
  high dimensional space.* ICDT — why a smaller/fractional p can stay discriminating in high dimensions.
- K. Beyer, J. Goldstein, R. Ramakrishnan, U. Shaft (1999). *When is "nearest neighbor" meaningful?*
  ICDT — distance concentration.
- P. C. Mahalanobis (1936). *On the generalised distance in statistics.* Proc. National Institute of
  Sciences of India, 2(1):49–55.
- T. Hastie, R. Tibshirani, J. Friedman (2009). *The Elements of Statistical Learning*, §13.3 (k-NN)
  and §7.10 (cross-validation). DOI:
  [10.1007/978-0-387-84858-7](https://doi.org/10.1007/978-0-387-84858-7).

---

*Previous: 05 — Demanding case: breast cancer & the curse* · *Next: Module 02 — Naive Bayes (chapter 01
complete)*